In [ ]:
# load required libraries
import sys
sys.path.append("../utils")
from pairing_utils import iou_batch
from json_parser import parse_json_annotations
from custom_coco_eval import evaluate_mask_rcnn
from custom_coco_eval import evaluate_detection_model, contruct_predictions_from_detections
from precision_recall_eval import evaluate_yolo_pr, evaluate_mask_rcnn_pr
import numpy as np
import os
import cv2
from typing import List, Dict, Union, Tuple, Final

# optical characteristics of the images (used to map the minimum diameter of objects in um to pixels)
OPTICAL_CHARACTERISTICS: Final[Dict[Tuple, Dict[str, float]]] = {(2000, 1600): {'mag': 10.0, 'pixel_size': 4.54}, 
                                                                 (4512, 4512): {'mag': 9.0, 'pixel_size': 2.74}}


In [ ]:
# the folders for test images and annotations
# they will be considered under the dataset folder specified by the class
# the images and annotations will share the same name
TEST_ANNOTATIONS_FOLDER = 'test_annotations'
TEST_IMAGES_FOLDER = 'test_images'

In [ ]:
def get_name(filename_with_ext: str):
    return ".".join(filename_with_ext.strip().split(".")[:-1])

class TestDataSet:
    def __init__(self,
                 dataset_paths: List[str],
                 scale_factor_dict: Dict[Tuple[int, int], float],
                 max_larger_side: int,
                 max_smaller_side: int,
                 class_names_to_ids_map: dict, 
                 labels_of_interest: Union[List[str], None] = None,
                 percentage_to_expand_bbox_boundaries: float = 0.0,
                 min_object_diameter: float = 6.0,
                 optical_characteristics: Dict[Tuple[int, int], Dict[str, float]] = OPTICAL_CHARACTERISTICS) -> None:
        """
        Args:
            dataset_paths (List[str]): List of dataset paths to be used for testing.
            scale_factor_dict (Dict[Tuple[int, int], float]):
            max_larger_side (int, optional): _description_. Defaults to 2000.
            max_smaller_side (int, optional): _description_. Defaults to 1600.
            class_names_to_ids_map (dict, optional):
            labels_of_interest (Union[List[str], None], optional): _description_. Defaults to None.
            percentage_to_expand_bbox_boundaries (float, optional): _description_. Defaults to 0.2.
            color_depth (int, optional): _description_. Defaults to 14.
            min_object_diameter (float, optional): _description_. Defaults to 0.0.
            optical_characteristics (Dict[Tuple[int, int], Dict[str, float]]): _description_. Defaults to
                OPTICAL_CHARACTERISTICS.
        """
        self.images_path: List[str] = []
        self.annotations_path: List[str] = []
        for test_folder in dataset_paths:
            
            annotations_files = os.listdir(os.path.join(test_folder, TEST_ANNOTATIONS_FOLDER))
            image_files =  os.listdir(os.path.join(test_folder, TEST_IMAGES_FOLDER))
            
            annotations_files_no_ext =  [get_name(file) for file in annotations_files]
            image_files_no_ext = [get_name(file) for file in image_files]
            
            annotations_files = [file for file in annotations_files if get_name(file) in image_files_no_ext]
            image_files = [file for file in image_files if get_name(file) in annotations_files_no_ext]

            if len(annotations_files) != len(annotations_files_no_ext) or len(image_files) != len(image_files_no_ext):
                print(f"[WARN] Found some unmatched images and anotations for dataset {test_folder}")

            self.annotations_path += [os.path.join(test_folder, TEST_ANNOTATIONS_FOLDER, file) for file in annotations_files]
            # make the two lists in the same order, here we assume the image extensions are always '.jpg'
            self.images_path += [os.path.join(test_folder, TEST_IMAGES_FOLDER, get_name(file) + '.jpg') for file in annotations_files]
        
        self.labels_of_interest = labels_of_interest
        # in order to reduce the memory required for the masks (for instance segmentation models)
        # the image and the annotations can be downsized by the passed scale_factor_dict
        # this is a dictionary with keys as the image resolutions for which the scaling should be
        # applied and values as the scaling factor
        # (values >= 1 are expected to downsize the image by the factor)
        self.scale_factor_dict = scale_factor_dict
        # the image is further resized (after applying the passed scale_factor above) to have the
        # larger side and the smaller side both smaller than these two maximum set values
        self.max_larger_side = max_larger_side
        self.max_smaller_side = max_smaller_side
        self.class_names_to_ids_map = class_names_to_ids_map
        self.percentage_to_expand_bbox_boundaries = percentage_to_expand_bbox_boundaries
        # minimum diameter of objects to keep in micro meter, objects smaller than this
        # minimum diameter are expected from the returned annotations
        # only 2000x1600 and 4512x4512 image resolutions are supported as this
        # diameter should be converted to a number of pixels based on magnification and sensor pixel size
        self.min_object_diameter = min_object_diameter
        # the optical characteristics of the device (used to convert the above min_object_diameter from um to pixels)
        self.optical_characteristics = optical_characteristics      

    def __getitem__(self, idx: int):
        # load images and masks
        annotation_path: str = self.annotations_path[idx]
        img_path: str = self.images_path[idx]

        if annotation_path[-4:] == 'json':
            # parse the annotations file, this is the first time an image/annotation pair is called
            annotations = parse_json_annotations(json_filename=annotation_path,
                                                 labels_of_interest=self.labels_of_interest,
                                                 download_image=False,
                                                 percentage_to_expand_bbox_boundaries=self.percentage_to_expand_bbox_boundaries,
                                                 min_object_diameter=self.min_object_diameter,
                                                 optical_characteristics=self.optical_characteristics,
                                                 return_masks_in_coco_rle_format=False)
            if self.class_names_to_ids_map is not None:
                annotations['annotations']['label'] = annotations['annotations']['label'].map(self.class_names_to_ids_map)
        elif annotation_path[-3:] == 'npy':
            # annotations in .npy format
            # note: annotations['annotations']['label'] is already int and no need to convert to integers
            annots = np.load(annotation_path, allow_pickle=True)
            annotations = {"name": os.path.basename(img_path), 
                           "annotations": annots.item().get("annotations"), 
                           "masks": annots.item().get("masks")}
            

        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        
        image_height, image_width = img.shape[:2]
        # scale factor
        if (image_width, image_height) in self.scale_factor_dict:
            scale_factor: float = self.scale_factor_dict[(image_width, image_height)]
        else:
            scale_factor: float = 1.0

        larger_side: int = max(int(image_width / scale_factor), int(image_height / scale_factor))
        smaller_side: int = min(int(image_width / scale_factor), int(image_height / scale_factor))

        if larger_side > self.max_larger_side or smaller_side > self.max_smaller_side:
            scale_factor *= max(float(larger_side) / self.max_larger_side, float(smaller_side) / self.max_smaller_side)

        if scale_factor != 1:
            # for decimating an image, cv2.INTER_AREA is the preferred method (scale_factor is always > 1)
            img = cv2.resize(img, (int(image_width / scale_factor), int(image_height / scale_factor)),
                             interpolation=cv2.INTER_AREA)
            # update all the masks and annotations
            annotations['annotations'][['xtl', 'ytl', 'xbr', 'ybr']] = annotations['annotations'][
                ['xtl', 'ytl', 'xbr', 'ybr']].div(scale_factor).astype(int)

            # make sure no box width/height becomes zero after the resize
            # only keep boxes with positive width and height
            annotations['annotations'] = annotations['annotations'][
                (annotations['annotations']['ybr'] - annotations['annotations']['ytl'] > 0) &
                (annotations['annotations']['xbr'] - annotations['annotations']['xtl'] > 0)]

            # keep the corresponding masks after resizing
            annotations['masks'] = [annotations['masks'][i] for i in annotations['annotations'].index]
            # reset the index
            annotations['annotations'].reset_index(inplace=True, drop=True)

            # now resize the masks (note that they are defined within the bounding boxes)
            for idx in range(len(annotations['masks'])):
                box_xtl, box_ytl, box_xbr, box_ybr = annotations['annotations'].loc[
                    idx, ['xtl', 'ytl', 'xbr', 'ybr']].values
                annotations['masks'][idx] = cv2.resize(annotations['masks'][idx],
                                                       (box_xbr - box_xtl, box_ybr - box_ytl),
                                                       interpolation=cv2.INTER_NEAREST)

        annotations['image'] = img
        # update the name of the image
        annotations['name']: str = os.path.basename(img_path)

        return annotations

    def __len__(self):
        return len(self.annotations_path)

In [ ]:
label_map = {1: 'cell', 2: 'bead', 3: 'cage', 4: 'nucleus', 5: 'cell-adhered'}
reverse_label_map = {'cell': 1, 'Cell': 1, 'dying/dead cells': 1, 
                     'Bead': 2, 'bead': 2, 
                     'cages': 3, 'cage': 3,
                     'nucleus': 4, 
                     'cytoplasm': 5, 'cell-adhered':5
                    }

dataset = TestDataSet(dataset_paths = ['/home/cellareye/Cellanome/Data/old_microscope_data_and_old_analysis_data_sets_1_2_test_set',
                                       '/home/cellareye/Cellanome/Data/imr_90_nucleus_cytoplasm_sets_1_2',
                                       '/home/cellareye/Cellanome/Data/imr_90_nucleus_cytoplasm_cage_set_3',
                                       '/home/cellareye/Cellanome/Data/231212_imr90_multichannel_overlay', 
                                       '/home/cellareye/Cellanome/Data/240213_imr90_multichannel_overlay',
                                       '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_caged',
                                       '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_uncaged'
                                      ], 
                      scale_factor_dict = {}, 
                      max_larger_side = 5000,
                      max_smaller_side = 5000,
                      class_names_to_ids_map = reverse_label_map,  
                      labels_of_interest = list(reverse_label_map.keys()))

In [ ]:
len(dataset)

In [ ]:
from mask_rcnn_model import run_mask_rcnn

In [ ]:
predictions: List = []
run_times: List = []
for idx in range(len(dataset)):
    annots = dataset[idx]
    
    preds, run_time = run_mask_rcnn(annots['image'], 
                                    normalize_image = False, 
                                    bit_depth = 8, 
                                    crop = True, 
                                    post_process_class_names = ['cell', 'nucleus', 'cage', 'cell-adhered'], 
                                    plot_results = False)
    
    run_times.append(run_time)
    
    # no need to filter for labels_of_interest here as we are doing it during precision and 
    # recall evaluation, similary, no need to only return the classes of interest from the
    # dataset
    
    predictions.append(
        {'boxes': np.array(preds['boxes']) if len(preds['boxes']) > 0 else np.zeros((0, 4), dtype=int), 
         'labels': np.array(preds['labels']) if len(preds['labels']) > 0 else np.zeros((0,), dtype=int), 
         'scores': np.array(preds['scores']) if len(preds['scores']) > 0 else np.zeros((0,), dtype=float), 
         'masks': preds['masks'], })
print(f"Running Mask RCNN took {np.mean(run_times[1:]) * 1000}ms on average per image")

In [ ]:
# None means all, 
# labels_of_interest = None
labels_of_interest = ['cell']

if labels_of_interest is None:
    # consider all classes/labels
    class_ids_of_interest = list(set(dataset.class_names_to_ids_map.values()))
else:
    class_ids_of_interest = list(set([dataset.class_names_to_ids_map[l] for l in labels_of_interest]))

# we can use the same function for calculating precision and recall of the Mask RCNN
# detections (here, we are not using any mask measure)
# precision, recall = evaluate_yolo_pr(predictions, dataset, class_ids_of_interest, 0.5)
precision, recall = evaluate_mask_rcnn_pr(predictions, dataset, class_ids_of_interest, 0.5)
ll_str = 'All' if  labels_of_interest is None else ', '.join(labels_of_interest)
print(f"Mask RCNN Precision: {precision}, Recall: {recall} at IoU 0.5 for labels: {ll_str}")
if precision + recall > 0:
    print(f"Mask RCNN F-1 Score: {2 * precision * recall / (precision + recall)} at IoU 0.5 for labels: {ll_str}")
    
# None means all, 
# labels_of_interest = None
labels_of_interest = ['bead']

if labels_of_interest is None:
    # consider all classes/labels
    class_ids_of_interest = list(set(dataset.class_names_to_ids_map.values()))
else:
    class_ids_of_interest = list(set([dataset.class_names_to_ids_map[l] for l in labels_of_interest]))

precision, recall = evaluate_mask_rcnn_pr(predictions, dataset, class_ids_of_interest, 0.5)
ll_str = 'All' if  labels_of_interest is None else ', '.join(labels_of_interest)
print(f"Mask RCNN Precision: {precision}, Recall: {recall} at IoU 0.5 for labels: {ll_str}")
if precision + recall > 0:
    print(f"Mask RCNN F-1 Score: {2 * precision * recall / (precision + recall)} at IoU 0.5 for labels: {ll_str}")
    
# None means all, 
# labels_of_interest = None
labels_of_interest = ['cage']

if labels_of_interest is None:
    # consider all classes/labels
    class_ids_of_interest = list(set(dataset.class_names_to_ids_map.values()))
else:
    class_ids_of_interest = list(set([dataset.class_names_to_ids_map[l] for l in labels_of_interest]))

precision, recall = evaluate_mask_rcnn_pr(predictions, dataset, class_ids_of_interest, 0.5)
ll_str = 'All' if  labels_of_interest is None else ', '.join(labels_of_interest)
print(f"Mask RCNN Precision: {precision}, Recall: {recall} at IoU 0.5 for labels: {ll_str}")
if precision + recall > 0:
    print(f"Mask RCNN F-1 Score: {2 * precision * recall / (precision + recall)} at IoU 0.5 for labels: {ll_str}")
    
    
labels_of_interest = ['nucleus']

if labels_of_interest is None:
    # consider all classes/labels
    class_ids_of_interest = list(set(dataset.class_names_to_ids_map.values()))
else:
    class_ids_of_interest = list(set([dataset.class_names_to_ids_map[l] for l in labels_of_interest]))

precision, recall = evaluate_mask_rcnn_pr(predictions, dataset, class_ids_of_interest, 0.5)
ll_str = 'All' if  labels_of_interest is None else ', '.join(labels_of_interest)
print(f"Mask RCNN Precision: {precision}, Recall: {recall} at IoU 0.5 for labels: {ll_str}")
if precision + recall > 0:
    print(f"Mask RCNN F-1 Score: {2 * precision * recall / (precision + recall)} at IoU 0.5 for labels: {ll_str}")

labels_of_interest = ['cell-adhered']

if labels_of_interest is None:
    # consider all classes/labels
    class_ids_of_interest = list(set(dataset.class_names_to_ids_map.values()))
else:
    class_ids_of_interest = list(set([dataset.class_names_to_ids_map[l] for l in labels_of_interest]))

precision, recall = evaluate_mask_rcnn_pr(predictions, dataset, class_ids_of_interest, 0.5)
ll_str = 'All' if  labels_of_interest is None else ', '.join(labels_of_interest)
print(f"Mask RCNN Precision: {precision}, Recall: {recall} at IoU 0.5 for labels: {ll_str}")
if precision + recall > 0:
    print(f"Mask RCNN F-1 Score: {2 * precision * recall / (precision + recall)} at IoU 0.5 for labels: {ll_str}")

In [ ]:
Mask RCNN Precision: 0.7489711934156379, Recall: 0.8625592417061612 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.801762114537445 at IoU 0.5 for labels: cell
Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead
Mask RCNN Precision: 0.9612590799031477, Recall: 0.9900249376558603 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9754299754299754 at IoU 0.5 for labels: cage
Mask RCNN Precision: 0.8589341692789969, Recall: 0.8509316770186336 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8549141965678628 at IoU 0.5 for labels: nucleus
Mask RCNN Precision: 0.8073089700996677, Recall: 0.8350515463917526 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8209459459459459 at IoU 0.5 for labels: cell-adhered